In [1]:
import numpy as np
import cv2
import time
import matplotlib.pyplot as plt
from PIL import Image
import os
from paddle.vision.transforms import functional as F


In [2]:
def get_color_map_list(num_classes):
    """
    Returns the color map for visualizing the segmentation mask,
    which can support arbitrary number of classes.
    Args:
        num_classes (int): Number of classes.
    Returns:
        (list). The color map.
    """

    num_classes += 1
    color_map = num_classes * [0, 0, 0]
    for i in range(0, num_classes):
        j = 0
        lab = i
        while lab:
            color_map[i * 3] |= (((lab >> 0) & 1) << (7 - j))
            color_map[i * 3 + 1] |= (((lab >> 1) & 1) << (7 - j))
            color_map[i * 3 + 2] |= (((lab >> 2) & 1) << (7 - j))
            j += 1
            lab >>= 3
    color_map = color_map[3:]
    return color_map

In [3]:
color_map = get_color_map_list(256)

In [4]:
src_img_path = "C:\\Users\\hp\\Desktop\\4_rect_1\\1024\\leftImg8bit"
src_mask_path = "C:\\Users\\hp\\Desktop\\4_rect_1\\1024\\gtFine"
src_bg_path = "C:\\Users\\hp\\Desktop\\4_rect_1\\bg\\1024" # 背景图片名字必须与原图片一致

save_img_path = "C:\\Users\\hp\\Desktop\\4_aug\\leftImg8bit"
save_mask_path = "C:\\Users\\hp\\Desktop\\4_aug\\gtFine"

In [5]:
imgs_name = []
for _item in os.listdir(src_img_path):
    if _item.split('.')[-1] in ['jpg', 'png', 'bmp', 'jpeg']:
        imgs_name.append(_item)

In [6]:
# [rowStart, rowEnd, colStart, colEnd]
fg_dict = {'1.jpg':[[295, 1299, 231, 1105]], '2.jpg':[[81,1283,317, 999]], '3.jpg':[[181,1275,99,1231]], 
           '4.jpg':[[]], '5.jpg':[[235,1107,293,1145]], '6.jpg':[[189, 1200, 203, 1085]], 
           '7.jpg':[[265, 1075, 189, 1015]], '8.jpg':[[275,1143,185,1063]], '9.jpg':[[215,1103, 303, 1169]], 
           '10.jpg':[[313,1205,165,1013]], '11.jpg':[[217,1281,95,1163]], '12.jpg':[[343,1037,323,995]], 
           '13.jpg':[[345,1213,155,1025]], '14.jpg':[[227,1127,187,1047]], '15.jpg':[[125,1200,105,1175]],
           '16.jpg':[[289,1137,200,1085]], '17.jpg':[[]]}

In [7]:
size = 1024
for key in fg_dict.keys():
    _plist = fg_dict[key]
    if _plist == [[]]:
        continue
    _new = [[0,0,0,0]]
    _new[0][0] = int( float(_plist[0][0])/1311.0 * size )
    _new[0][1] = int( float(_plist[0][1])/1311.0 * size )
    _new[0][2] = int( float(_plist[0][2])/1361.0 * size )
    _new[0][3] = int( float(_plist[0][3])/1361.0 * size )
    fg_dict[key] = _new

In [8]:
fg_dict_1024 = {'18.jpg':[[28,855, 137,925]],
           '19.jpg':[[190, 833, 189, 806]], '20.jpg':[[53,867,18,802]], '21.jpg':[[71,1023, 18,985]],
           '22.jpg':[[82, 821, 149,857]], '23.jpg':[[0,508,239, 939]], '24.jpg':[[131,853, 100,807]],
           '25.jpg':[[131,853, 100,807]], '26.jpg':[[186,902,204,895]], '27.jpg':[[231,918,103,781]],
           '28.jpg':[[53,1131,37,1157]],'29.jpg':[[139,971,51,835]],'30.jpg':[[81,971,147,975]],
           '31.jpg':[[105,953,117,979]],'32.jpg':[[137,905,161,971]],'33.jpg':[[]],
           '34.jpg':[[35,867,5,779]],'35.jpg':[[51,901,5,783]],'36.jpg':[[191,1021,169,1021]],
           '37.jpg':[[27,941,19,937]],'38.jpg':[[175,949,129,931]],'39.jpg':[[93,843,82,880]],
           '40.jpg':[[0,1023,0,1023]],'41.jpg':[[0,1023,0,1023]],'42.jpg':[[0,1023,0,1023]],
           '43.jpg':[[0,1023,0,1023]], '44.jpg':[[99,840,214,949]]}

In [9]:
fg_dict.update(fg_dict_1024)

In [10]:
print(fg_dict)

{'1.jpg': [[230, 1014, 173, 831]], '2.jpg': [[63, 1002, 238, 751]], '3.jpg': [[141, 995, 74, 926]], '4.jpg': [[]], '5.jpg': [[183, 864, 220, 861]], '6.jpg': [[147, 937, 152, 816]], '7.jpg': [[206, 839, 142, 763]], '8.jpg': [[214, 892, 139, 799]], '9.jpg': [[167, 861, 227, 879]], '10.jpg': [[244, 941, 124, 762]], '11.jpg': [[169, 1000, 71, 875]], '12.jpg': [[267, 809, 243, 748]], '13.jpg': [[269, 947, 116, 771]], '14.jpg': [[177, 880, 140, 787]], '15.jpg': [[97, 937, 79, 884]], '16.jpg': [[225, 888, 150, 816]], '17.jpg': [[]], '18.jpg': [[28, 855, 137, 925]], '19.jpg': [[190, 833, 189, 806]], '20.jpg': [[53, 867, 18, 802]], '21.jpg': [[71, 1023, 18, 985]], '22.jpg': [[82, 821, 149, 857]], '23.jpg': [[0, 508, 239, 939]], '24.jpg': [[131, 853, 100, 807]], '25.jpg': [[131, 853, 100, 807]], '26.jpg': [[186, 902, 204, 895]], '27.jpg': [[231, 918, 103, 781]], '28.jpg': [[53, 1131, 37, 1157]], '29.jpg': [[139, 971, 51, 835]], '30.jpg': [[81, 971, 147, 975]], '31.jpg': [[105, 953, 117, 979]], '

In [11]:
def rotate_and_resize_roi(img1, theta, scale_ratio=1):
    img = img1.copy()
    _rotateCenter = (np.shape(img)[1]//2, np.shape(img)[0]//2)#旋转中心
    _img_size = (np.shape(img)[1], np.shape(img)[0])#变换后的大小
    
    _R = cv2.getRotationMatrix2D(_rotateCenter, theta, scale_ratio) #计算旋转的仿射变换矩阵
    img_rotate = cv2.warpAffine(img, _R, _img_size)
    return theta, scale_ratio, img_rotate

In [14]:
# 原图片区域移动、旋转与放缩
aug_num = 10
angles_list = np.linspace(1, 30, 30) # 微旋转
angles_list_no = np.linspace(0,0,60)
angles_list = np.hstack([angles_list, angles_list_no])
angles_list = angles_list.astype(np.int32)

scale_list = np.linspace(0.8,2,60) # 大部分放大，小部分缩小
scale_list_no = np.linspace(1,1,60)
scale_list = np.hstack((scale_list, scale_list_no))

for _imgName in imgs_name:
    _maskName = _imgName.split('.')[0] + '.png'
    
    for i in range(aug_num):
        img = Image.open(os.path.join(src_img_path, _imgName))#加载图片
        img_bg = Image.open(os.path.join(src_bg_path, _imgName))#加载图片
        img_mask = Image.open(os.path.join(src_mask_path, _maskName))
        
        img_bg = np.array(img_bg)
        img = np.array(img)
        rect = fg_dict[_imgName][0]
        if rect == []:
            break

        img_roi = img[rect[0]:rect[1], rect[2]:rect[3], :]
        _theta, _scale, img_rotate = rotate_and_resize_roi(img_roi, 
                                                theta = np.random.choice(angles_list), 
                                                scale_ratio=np.random.choice(scale_list))
        img_roi_bg = img_bg[rect[0]:rect[1], rect[2]:rect[3], :]
        _img2_arr_mask1 = np.logical_and(img_rotate[:,:,0] == 0 , img_rotate[:,:,1] == 0) # 求R和G通道bool矩阵的交集
        img2_arr_mask = np.logical_and(_img2_arr_mask1, img_rotate[:,:,2] == 0)
        img_rotate[img2_arr_mask] = img_roi_bg[img2_arr_mask]
        
        # img_copy = img.copy()
        h,w,c = img_rotate.shape
        img[rect[0]:rect[1], rect[2]:rect[3], :] = img_bg[rect[0]:rect[1], rect[2]:rect[3], :] #原图片区域覆盖
        row_range = np.linspace(0,img.shape[0]-h,20)
        row_range = row_range.astype(np.int32)
        col_range = np.linspace(0,img.shape[1]-w,20)
        col_range = col_range.astype(np.int32)
        start_point = [np.random.choice(row_range), np.random.choice(col_range)]
        img[start_point[0]:start_point[0]+h, start_point[1]:start_point[1]+w] = img_rotate
                
        # 保存新图片
        _new_img_name = _imgName.split('.')[0]+'_'+str(i)+'trans'+'.'+_imgName.split('.')[-1]
        converted_img = Image.fromarray(img)
        converted_img.save(os.path.join(save_img_path, _new_img_name))

        # 处理掩膜
        img_mask = np.array(img_mask)
        img_mask_roi = img_mask[rect[0]:rect[1], rect[2]:rect[3]]
        _th,_sc, img_mask_rotate = rotate_and_resize_roi(img_mask_roi, _theta, scale_ratio=_scale)
        
        h,w = img_mask_rotate.shape
        img_mask[rect[0]:rect[1], rect[2]:rect[3]] = 0 #原掩膜图片区域覆盖，背景像素标签为0
        img_mask[start_point[0]:start_point[0]+h, start_point[1]:start_point[1]+w] = img_mask_rotate
        #保存标签
        lbl_pil = Image.fromarray(img_mask)
        lbl_pil.putpalette(color_map)
        _new_mask_name = _maskName.split('.')[0]+'_'+str(i)+'trans'+'.'+_maskName.split('.')[-1]
        lbl_pil.save(os.path.join(save_mask_path, _new_mask_name))
    
    

In [ ]:
# 以下是单张图片的测试

In [ ]:
img_name = '16.jpg'
img = Image.open(os.path.join(src_img_path, img_name))#加载图片
img_bg = Image.open(os.path.join(src_bg_path, img_name))#加载图片
plt.figure(figsize=(6,12))
plt.subplot(1,2,1)
plt.imshow(img)
plt.subplot(1,2,2)
plt.imshow(img_bg)

In [ ]:
img_bg = np.array(img_bg)
img = np.array(img)
rect = fg_dict[img_name][0]

img_roi = img[rect[0]:rect[1], rect[2]:rect[3], :]
_theta, _scale, img_rotate = rotate_and_resize_roi(img_roi, 
                                        theta = np.random.choice(angles_list), scale_ratio=np.random.choice(scale_list))
img_roi_bg = img_bg[rect[0]:rect[1], rect[2]:rect[3], :]
_img2_arr_mask1 = np.logical_and(img_rotate[:,:,0] == 0 , img_rotate[:,:,1] == 0) # 求R和G通道bool矩阵的交集
img2_arr_mask = np.logical_and(_img2_arr_mask1, img_rotate[:,:,2] == 0)
img_rotate[img2_arr_mask] = img_roi_bg[img2_arr_mask]
plt.figure(figsize=(12,12))
plt.subplot(1,2,1)
plt.imshow(img_roi)
plt.subplot(1,2,2)
plt.imshow(img_rotate)

In [ ]:
img_copy = img.copy()
h,w,c = img_rotate.shape
img[rect[0]:rect[1], rect[2]:rect[3], :] = img_bg[rect[0]:rect[1], rect[2]:rect[3], :] #原图片区域覆盖
row_range = np.linspace(0,img.shape[0]-h,20)
row_range = row_range.astype(np.int32)
col_range = np.linspace(0,img.shape[1]-w,20)
col_range = col_range.astype(np.int32)
start_point = [np.random.choice(row_range), np.random.choice(col_range)]
img[start_point[0]:start_point[0]+h, start_point[1]:start_point[1]+w] = img_rotate
plt.figure(figsize=(12,12))
plt.subplot(1,2,1)
plt.imshow(img)
plt.subplot(1,2,2)
plt.imshow(img_copy)

In [ ]:
img_mask_name = '16.png'
img_mask = Image.open(os.path.join(src_mask_path, img_mask_name))#加载图片
img_mask = np.array(img_mask)
rect = fg_dict[img_name][0]

img_mask_roi = img_mask[rect[0]:rect[1], rect[2]:rect[3]]
_th,_sc, img_mask_rotate = rotate_and_resize_roi(img_mask_roi, _theta, scale_ratio=_scale)
plt.figure(figsize=(12,12))
plt.subplot(1,2,1)
plt.imshow(img_mask_roi)
plt.subplot(1,2,2)
plt.imshow(img_mask_rotate)

In [ ]:
img_mask_copy = img_mask.copy()
h,w = img_mask_rotate.shape
img_mask[rect[0]:rect[1], rect[2]:rect[3]] = 0 #原掩膜图片区域覆盖，背景像素标签为0

img_mask[start_point[0]:start_point[0]+h, start_point[1]:start_point[1]+w] = img_mask_rotate
plt.figure(figsize=(12,12))
plt.subplot(1,2,1)
plt.imshow(img_mask)
plt.subplot(1,2,2)
plt.imshow(img_mask_copy)